# V and J genes from a CDR3: posterior and calibrated candidate set

Companion notebook to *Quantifying the Recoverability of V and J Genes from TCR CDR3
Sequences Using Generative Repertoire Models* (Huang & Baras).

Enter a CDR3 amino acid sequence and a chain, and this returns:

- the posterior over V genes and over J genes, as barplots
- the **calibrated candidate set**: the smallest set of genes that contains the true gene
  with a requested probability, using the coverage calibration measured in the paper
  rather than nominal posterior mass
- the gene's confusion group, where it has one

The posterior is computed by the same code as the paper
(`supervdj.posterior.preselection_posterior`), so the numbers here are the paper's
numbers, not a reimplementation.

**What it cannot do.** The posterior is conditioned on the CDR3 amino acid sequence alone,
under the OLGA recombination model. J is recovered almost completely; V usually is not,
and for the beta chain a single best call is correct about a fifth of the time. Where
CDR1, CDR2, framework or paired-chain sequence is available, standard IMGT annotation is
the better choice. Read the candidate set, not the top call.

## Setup (about a minute)

In [ ]:
import os, subprocess, sys

REPO = "https://github.com/samhuann/SuperVDJMachine"
if not os.path.isdir("SuperVDJMachine"):
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "olga", "matplotlib", "pandas"],
               check=True)
sys.path.insert(0, "SuperVDJMachine")
print("ready")

In [ ]:
from functools import lru_cache

import matplotlib.pyplot as plt
import pandas as pd

from supervdj.cache import ValueCache
from supervdj.models import load_chain_models
from supervdj.posterior import preselection_posterior

CAL = "SuperVDJMachine/results/confusion_groups"


@lru_cache(maxsize=2)
def models(chain):
    """OLGA only; the selection model is not needed for the pre-selection posterior."""
    return load_chain_models(chain, use_sonia=False), ValueCache(None)


@lru_cache(maxsize=2)
def calibration(chain):
    """Measured coverage -> nominal posterior mass needed to achieve it (paper Table)."""
    t = pd.read_csv(f"{CAL}/coverage_targets_{chain}.tsv", sep="\t")
    g = pd.read_csv(f"{CAL}/confusion_groups_{chain}.tsv", sep="\t")
    return t, dict(zip(g.gene, g.group))


def posteriors(cdr3, chain):
    m, c = models(chain)
    v = preselection_posterior(m, c, cdr3, "V")
    j = preselection_posterior(m, c, cdr3, "J")
    if not v:
        raise ValueError(
            f"{cdr3!r} has zero generation probability under the {chain} model: the model "
            "treats it as impossible, so no posterior is defined. Check the CDR3 begins "
            "with C and ends in F or W, and that the chain is right.")
    return v, j


def candidate_set(post, chain, coverage=0.9):
    """Smallest set reaching the nominal mass that the paper measured to give `coverage`.

    Asking for 0.9 coverage is not the same as taking genes until the posterior sums to
    0.9; the nominal mass needed was calibrated on held-out data and is usually higher.
    """
    targets, _ = calibration(chain)
    row = targets.iloc[(targets.target_coverage - coverage).abs().argsort().iloc[0]]
    need = float(row.nominal_mass_needed)
    out, cum = [], 0.0
    for g, m in sorted(post.items(), key=lambda kv: -kv[1]):
        out.append((g, m))
        cum += m
        if cum >= need:
            break
    return out, float(row.target_coverage), need


def report(cdr3, chain="TRB", coverage=0.9, top=12):
    cdr3 = cdr3.strip().upper()
    v, j = posteriors(cdr3, chain)
    _, groups = calibration(chain)

    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    for a, post, name in ((ax[0], v, "V"), (ax[1], j, "J")):
        top_n = sorted(post.items(), key=lambda kv: -kv[1])[:top]
        a.barh([g for g, _ in top_n][::-1], [m for _, m in top_n][::-1], color="#4C72B0")
        a.set_xlabel(f"P({name} gene | CDR3)")
        a.set_xlim(0, 1)
        a.set_title(f"{chain} {name}  (top {len(top_n)} of {len(post)})")
    fig.suptitle(f"{cdr3}   {chain}", y=1.02)
    fig.tight_layout()
    plt.show()

    for post, name in ((v, "V"), (j, "J")):
        cset, cov, need = candidate_set(post, chain, coverage)
        best, bm = max(post.items(), key=lambda kv: kv[1])
        print(f"{chain} {name}")
        print(f"  best single call   {best} at posterior mass {bm:.3f}")
        print(f"  calibrated set for {cov:.0%} coverage "
              f"(nominal mass {need:.3f}): {len(cset)} gene(s)")
        print("    " + ", ".join(f"{g} {m:.3f}" for g, m in cset))
        if name == "V" and best in groups:
            mates = sorted(g for g, k in groups.items()
                           if k == groups[best] and g != best)
            print(f"  confusion group of {best}: "
                  + (", ".join(mates) if mates else "no other gene"))
        print()
    return v, j


print("helpers defined")

## Try it

Edit `CDR3` and `CHAIN` and run. `CHAIN` is `"TRB"` or `"TRA"`.

In [ ]:
CDR3 = "CASSLMTNQPQHF"   # a beta CDR3 from VDJdb
CHAIN = "TRB"
COVERAGE = 0.9           # how often the set should contain the true gene

v, j = report(CDR3, CHAIN, COVERAGE)

## The asymmetry the paper is about

Run a few CDR3s and compare how large the V set has to be against the J set.

In [ ]:
import math

EXAMPLES = [("CASSLMTNQPQHF", "TRB"),
            ("CASSLYGSVQNEQFF", "TRB"),
            ("CAAADDKIIF", "TRA"),
            ("CAVRDSNYQLIW", "TRA")]

rows = []
for cdr3, chain in EXAMPLES:
    try:
        v, j = posteriors(cdr3, chain)
    except ValueError as e:
        print(e)
        continue
    ent = lambda p: -sum(m * math.log(m) for m in p.values() if m > 0)
    rows.append({"CDR3": cdr3, "chain": chain,
                 "V set @90%": len(candidate_set(v, chain, 0.9)[0]),
                 "J set @90%": len(candidate_set(j, chain, 0.9)[0]),
                 "V entropy": round(ent(v), 3), "J entropy": round(ent(j), 3)})
pd.DataFrame(rows)

The J set is usually one gene and the V set is several. That gap, measured across 118,096
rearrangements and replicated in three independent cohorts including a healthy bulk
repertoire, is the result the paper reports.

For batch use, the command-line pipeline in the repository is faster than this notebook:

```
PYTHONPATH=. python -m supervdj.run --help
```